# Pipeline multi-agente — UPV-EARTH (Cascada de **enriquecimiento**)

**Objetivo**: superar el baseline `qwen2.5:14b` v4 (Hit@K > 72 %) **en calidad**, no sólo en coste.

## ¿Por qué la versión anterior podía empeorar el baseline?

Una cascada que sólo **filtra** (Agente pequeño decide si pasar al grande, sí/no) **no puede mejorar** la precisión del baseline: en el mejor caso conserva igual recall y baja coste; en el peor pierde papers válidos por falsos negativos del filtro. Y depende de `keywords` / `journal` que están NaN en 44 % / 11 % del corpus.

## Idea nueva: **enriquecimiento, no filtrado**

| Etapa | Modelo | Cambio respecto a la v1 del pipeline |
|---|---|---|
| **Agente 1** | `qwen2.5:3b` | Ya no es filtro binario. Ahora es **extractor estructurado**: separa químicos / físicos / biológicos / metodología / marco disciplinar |
| **Scorer determinista** | Python puro | NUEVO. Cruza abstract + title + top_terms con `core_keywords`, `extended_keywords`, `applied_keywords_upv` de cada PB del `pb_reference.csv`. **No depende de keywords/journal del corpus** |
| **Agente 3** | `qwen2.5:14b` | Recibe **3 señales** en vez de 1: extracción semántica + ranking determinista + texto crudo. Mismo prompt v4 (principle-driven) |
| **Router** | Python | Consenso 3-de-3 (no binario). Sólo `fast_skip` si los TRES indicadores coinciden en "no biofísico" → drásticamente menos falsos negativos |

## Por qué esto sí supera al baseline

1. **El 14B siempre se ejecuta** en papers dudosos → cero pérdida de recall vs. baseline.
2. **El 14B recibe información que no tenía antes**: el scorer determinista usa el vocabulario oficial del `pb_reference.csv` (las 6 columnas de keywords), que el baseline ignora.
3. **Los papers sin keywords sí se procesan bien**: el scorer trabaja sobre el abstract + title + top_terms, no sólo sobre `keywords`.
4. **Coste sólo se ahorra cuando es seguro**: el `fast_skip` exige acuerdo 3-de-3.

El equivalente "producción" está en [`pipeline_agentes.py`](pipeline_agentes.py).

## 1. Imports y rutas

Reutilizamos la convención de los runners (`ROOT_DIR` = raíz del repo, `LLM_DIR` = `nlp/llm`).

In [ ]:
from __future__ import annotations
import json, re, time, logging, sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ROOT_DIR = Path.cwd().resolve()
while ROOT_DIR.name and not (ROOT_DIR / 'data' / 'corpus').exists():
    if ROOT_DIR.parent == ROOT_DIR:
        raise RuntimeError('No encuentro la raíz del proyecto (que contiene data/corpus).')
    ROOT_DIR = ROOT_DIR.parent
LLM_DIR = ROOT_DIR / 'nlp' / 'llm'
OUTPUTS_DIR = LLM_DIR / 'outputs' / 'pipeline_cascada'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = ROOT_DIR / 'data' / 'corpus' / 'master_corpus_mixto_1000_clean_enriched.csv'
PB_REF_PATH = ROOT_DIR / 'corpus_PB' / 'data' / 'pb_reference.csv'
GT_PATH     = LLM_DIR / 'outputs' / 'ground_truth' / 'validacion_real.csv'

print('ROOT_DIR =', ROOT_DIR)
print('CORPUS   =', CORPUS_PATH.exists(), CORPUS_PATH)
print('PB_REF   =', PB_REF_PATH.exists(), PB_REF_PATH)
print('GT       =', GT_PATH.exists(), GT_PATH)

## 2. EDA del corpus enriched

Vamos a mirar qué columnas tiene, cuántos nulos hay y la distribución de longitudes del abstract.

Esto importa porque:
- Si `keywords` o `journal` están NaN, el Agente 1 recibe menos contexto y baja en precisión.
- Si el abstract es enorme (>8000 chars), hay que truncar para no inflar el `num_ctx` del modelo.
- `top_terms_no_stopwords` es un campo **precalculado** que podemos usar como **fallback** cuando faltan keywords.

In [ ]:
df_corpus = pd.read_csv(CORPUS_PATH)
print(f'Filas: {len(df_corpus):,}   Columnas: {len(df_corpus.columns)}')
print()
summary = pd.DataFrame({
    'dtype': df_corpus.dtypes.astype(str),
    'nulls': df_corpus.isna().sum(),
    'null_pct': (df_corpus.isna().sum() / len(df_corpus) * 100).round(1),
})
summary

In [ ]:
# Distribución de longitudes del campo que va al LLM (clean_abstract).
lens = df_corpus['clean_abstract'].dropna().str.len()
stats = lens.describe(percentiles=[.25, .5, .75, .9, .99]).round(0).astype(int)
print('Longitud de clean_abstract (caracteres):')
print(stats.to_string())
print(f'\n→ El P99 es {stats["99%"]} chars. Vamos a truncar a 8000 chars (~2k tokens) por seguridad.')

In [ ]:
# Veamos los journals más frecuentes — pista de la composición del corpus.
df_corpus['journal'].fillna('(NaN)').value_counts().head(15).to_frame('papers')

## 3. EDA del `pb_reference.csv`

Este CSV define los 9 boundaries. Tiene **17 columnas** pero el prompt maestro sólo usa 5 (`pb_code`, `pb_name`, `short_definition`, `applied_keywords_upv`, `activation_logic`, `exclusion_notes`). Vamos a inspeccionarlas para decidir qué meter en cada agente.

In [ ]:
df_pbs = pd.read_csv(PB_REF_PATH)
print(f'Filas: {len(df_pbs)}   Columnas: {len(df_pbs.columns)}')
print()
print('Columnas disponibles:')
for c in df_pbs.columns:
    sample = str(df_pbs[c].iloc[0])
    print(f'  - {c:35s} → {sample[:80]}{"..." if len(sample)>80 else ""}')

In [ ]:
# Vista compacta de los 9 PBs.
df_pbs[['pb_code', 'pb_name', 'short_definition']]

**Cómo usa cada componente las columnas del `pb_reference.csv`:**

- **Bloque `<reference_framework>` del Agente 3** (las "reglas PB"): `pb_code`, `pb_name`, `short_definition`, `applied_keywords_upv`, `activation_logic`, `exclusion_notes`. Igual que en el v4 baseline.
- **Scorer determinista (sección 9)**: `core_keywords` (peso 3) + `extended_keywords` (peso 2) + `applied_keywords_upv` (peso 1). **Esto es nuevo y es donde está el upside vs. el v4**: el baseline ignoraba `core_keywords` y `extended_keywords`.
- **Agente 1**: NO recibe el `pb_reference.csv`. Es agnóstico al boundary; sólo extrae estructura semántica del paper.

## 4. Carga del ground truth (208 papers validados)

Lo usamos para dos cosas:
1. **Filtrar el corpus** a los 208 papers con etiqueta humana → calibración limpia.
2. **Calcular métricas** de la cascada al final (matriz de confusión, ahorro real).

In [ ]:
df_gt = pd.read_csv(GT_PATH, sep=';', encoding='utf-8')
df_gt['doc_id'] = df_gt['doc_id'].astype(str)
print(f'Ground truth: {len(df_gt)} papers, columnas {list(df_gt.columns)}')

# Distribución de la etiqueta humana primaria
df_gt['1stpb_label'] = df_gt['1stpb'].apply(lambda x: 'None' if pd.isna(x) else f'PB{int(x)}')
print('\nDistribución de 1stpb (etiqueta humana primaria):')
print(df_gt['1stpb_label'].value_counts().to_string())
n_none = (df_gt['1stpb_label'] == 'None').sum()
print(f'\n→ {n_none}/{len(df_gt)} = {100*n_none/len(df_gt):.1f}% son "None" (potencial fast_skip).')
print('   Si el Agente 1 acierta al filtrarlos, ese % es el ahorro teórico.')

In [ ]:
# Filtramos el corpus a sólo los validados, para calibrar contra GT.
df_corpus['doc_id'] = df_corpus['doc_id'].astype(str)
df_sample = df_corpus[df_corpus['doc_id'].isin(df_gt['doc_id'])].copy()
print(f'Papers del corpus que están en el ground truth: {len(df_sample)}')
missing = set(df_gt['doc_id']) - set(df_corpus['doc_id'])
if missing:
    print(f'⚠️  {len(missing)} doc_ids del GT no aparecen en el corpus enriched (descartados).')

## 5. Verificar Ollama

Antes de pelearnos con prompts, comprobamos que Ollama responde y qué modelos están descargados.

In [ ]:
OLLAMA_URL = 'http://localhost:11434/api/generate'
OLLAMA_TAGS = 'http://localhost:11434/api/tags'

try:
    r = requests.get(OLLAMA_TAGS, timeout=5)
    r.raise_for_status()
    models = sorted(m['name'] for m in r.json().get('models', []))
    print('✅ Ollama responde. Modelos instalados:')
    for m in models:
        print(f'   • {m}')
except Exception as e:
    print(f'❌ Ollama no responde: {e}')
    print('   Arráncalo con: `ollama serve` y verifica con `ollama list`.')

In [ ]:
# Calibra aqui los modelos. Si falta qwen2.5:3b: `ollama pull qwen2.5:3b`.
AGENT1_MODEL = 'qwen2.5:3b'    # extractor estructurado
AGENT3_MODEL = 'qwen2.5:14b'   # juez experto

# Preflight: ABORTA si falta un modelo (asi no perdemos tiempo en 404s silenciosos).
missing = [m for m in [AGENT1_MODEL, AGENT3_MODEL] if m not in models]
if missing:
    raise RuntimeError(
        f'Modelos no instalados en Ollama: {missing}\n'
        f'Disponibles: {models}\n'
        f'Descargalos con: ' + ' && '.join(f'ollama pull {m}' for m in missing)
    )
print(f'Preflight OK: {AGENT1_MODEL} y {AGENT3_MODEL} disponibles.')


## 6. Cliente Ollama con retries y `keep_alive`

Tres detalles importantes:

1. **`keep_alive=-1`**: mantiene el modelo cargado en VRAM indefinidamente. Sin esto, Ollama lo descarga a los 5 min y la siguiente llamada paga ~5-10 s extra de carga.
2. **Retries con backoff**: si Ollama tose un 5xx, reintentamos 3 veces (2 s, 4 s, 8 s). No reintentamos sobre 4xx (errores de cliente como 404 = modelo no existe).
3. **Timeout grande**: `(connect=10s, read=600s)`. El 14B sobre CPU puede tardar minutos.

In [ ]:
# keep_alive='10m' = Ollama descarga el modelo no usado si necesita VRAM.
# -1 (indefinido) puede colgar el daemon si 14B + 3B no caben simultaneamente en GPU.
KEEP_ALIVE = '10m'
HTTP_TIMEOUT = (10, 600)
MAX_ABSTRACT_CHARS = 8000

class OllamaClient:
    def __init__(self, url=OLLAMA_URL, timeout=HTTP_TIMEOUT):
        self.url, self.timeout = url, timeout
        self.session = requests.Session()
        retry = Retry(total=3, backoff_factor=2.0,
                      status_forcelist=(500, 502, 503, 504),
                      allowed_methods=frozenset(['POST']),
                      raise_on_status=False)
        self.session.mount('http://', HTTPAdapter(max_retries=retry))

    def generate(self, model, prompt, json_mode=False, temperature=0.0, num_ctx=None):
        opts = {'temperature': temperature, 'top_p': 0.9}
        if num_ctx is not None:
            opts['num_ctx'] = num_ctx
        payload = {'model': model, 'prompt': prompt, 'stream': False,
                   'keep_alive': KEEP_ALIVE, 'options': opts}
        if json_mode:
            payload['format'] = 'json'
        t0 = time.perf_counter()
        r = self.session.post(self.url, json=payload, timeout=self.timeout)
        elapsed = time.perf_counter() - t0
        r.raise_for_status()
        return r.json().get('response', ''), elapsed

    def warmup(self, model):
        try:
            _, t = self.generate(model=model, prompt='ping')
            print(f'   warmup {model} OK ({t:.2f}s)')
        except Exception as e:
            print(f'   ⚠️ warmup {model} fallo: {e}')

client = OllamaClient()
print('Pre-cargando modelos en VRAM...')
client.warmup(AGENT1_MODEL)
client.warmup(AGENT3_MODEL)

## 7. Construcción dinámica de las reglas PB

Compactamos los 9 PBs en un bloque de texto que el Agente 3 verá en cada llamada.

In [ ]:
def build_pb_rules(df_pbs):
    rules = ''
    for _, row in df_pbs.iterrows():
        rules += f"- PB Code: {row['pb_code']} ({row['pb_name']})\n"
        rules += f"  * Core Definition: {row['short_definition']}\n"
        rules += f"  * UPV Context: Look for terms like: {row['applied_keywords_upv']}\n"
        rules += f"  * ACTIVATION LOGIC: {row['activation_logic']}\n"
        rules += f"  * EXCLUSION RULE (CRITICAL): {row['exclusion_notes']}\n\n"
    return rules

PB_RULES = build_pb_rules(df_pbs)
print(f'Bloque de reglas PB: {len(PB_RULES):,} caracteres (~{len(PB_RULES)//4:,} tokens).')
print('\n--- Primeros 600 chars ---')
print(PB_RULES[:600])

## 8. Agente 1 — Extractor estructurado (qwen2.5:3b)

**Cambio de filosofía**: el Agente 1 ya no decide si un paper "tiene PB" o no. Su rol ahora es **enriquecer**: extraer información estructurada que el 14B usará después.

### Esquema de salida (JSON forzado)

```json
{
  "chemical_species":      ["CO2", "PM2.5", "nitrogen"],
  "physical_metrics":      ["river flow", "soil moisture"],
  "biological_observations":["bird species richness"],
  "methodology":           "measured | modelled | mentioned | none",
  "disciplinary_frame":    "engineering | earth_sciences | biology | chemistry | social_sciences | economics | law_policy | education | other"
}
```

**Por qué estos 5 campos:**
- Los 3 primeros son **lo que mide el paper**, separado por tipo de variable (los 9 PBs se dividen entre estos tres tipos).
- `methodology` distingue **operational object** vs. mero contexto — exactamente el principio v4. Si todo es "mentioned", probablemente no hay PB.
- `disciplinary_frame` es un **prior disciplinar**: un paper de pure economics tiene baja probabilidad a priori de ser un PB real.

### ¿Y los falsos positivos del extractor?

Da igual: este output **no decide nada por sí mismo**, sólo informa al 14B y al router de consenso. Si el 3B inventa una métrica, el 14B lo verá al leer el abstract y la descartará.

In [ ]:
# Agente 1: extractor estructurado multi-campo, JSON forzado.
# IMPORTANTE: el prompt NO contiene ejemplos in-line de variables.
# Razon: pruebas con llama3.2:3b mostraron que el modelo pequeno copiaba los
# ejemplos del prompt como si fueran datos del paper (efecto loro).
# Defensa adicional: _grounding_filter descarta items que no aparecen en el abstract.

AGENT1_PROMPT = """<system_role>
You are a fast, structured information extractor inside a multi-agent pipeline. A larger expert model runs after you. Your output enriches the next model's context; you do NOT make the final decision.
</system_role>

<task>
Read the metadata + abstract. Extract structured biophysical information into 5 fields (defined below). Every item you list must be COPIED VERBATIM from the abstract or title text below — never from these instructions.
</task>

<rules>
1. ZERO HALLUCINATION: every string in chemical_species, physical_metrics or biological_observations must be a direct substring of the <abstract> or <title> in the input. If you cannot point to it in the abstract, do NOT include it.
2. Distinguish what the paper measures/models (use those fields) from what it merely mentions in passing (do NOT extract those).
3. The disciplinary_frame is the dominant academic angle of the abstract, not the topic it discusses.
4. Empty arrays are valid and EXPECTED for non-biophysical papers. Do NOT fill them to look productive.
</rules>

<field_definitions>
- chemical_species: any chemical compound, element, ion, pollutant, gas, mineral or molecular species named in the abstract.
- physical_metrics: any measured or modelled physical or environmental variable named in the abstract (climate, energy, flow, mass, radiation, geophysical processes, etc.).
- biological_observations: any living-system observation named in the abstract (organisms, populations, ecosystems, physiology, ecological responses, etc.).
- methodology: ONE of {{"measured", "modelled", "mentioned", "none"}}.
    - "measured": the abstract reports primary or secondary empirical data.
    - "modelled": the abstract builds, runs or evaluates a model that produces values.
    - "mentioned": variables appear only as background, motivation or citation.
    - "none": no biophysical variable appears at all.
- disciplinary_frame: ONE of {{"engineering", "earth_sciences", "biology", "chemistry", "social_sciences", "economics", "law_policy", "education", "other"}}.
</field_definitions>

<self_check>
Before producing JSON, mentally verify: each string in your three arrays appears VERBATIM somewhere in the abstract below. If not, drop it.
</self_check>

<input_data>
<title>{title}</title>
<journal>{journal}</journal>
<keywords>{keywords}</keywords>
<abstract>
{abstract}
</abstract>
</input_data>

<output_format>
Return ONLY valid JSON in this exact structure (no markdown, no preamble):
{{
  "chemical_species": [],
  "physical_metrics": [],
  "biological_observations": [],
  "methodology": "measured|modelled|mentioned|none",
  "disciplinary_frame": "engineering|earth_sciences|biology|chemistry|social_sciences|economics|law_policy|education|other"
}}
</output_format>"""

VALID_METHODOLOGY = {'measured', 'modelled', 'mentioned', 'none'}
VALID_FRAMES = {'engineering', 'earth_sciences', 'biology', 'chemistry',
                'social_sciences', 'economics', 'law_policy', 'education', 'other'}
NON_BIO_FRAMES = {'social_sciences', 'economics', 'law_policy', 'education'}

@dataclass
class Agent1Result:
    raw_output: str
    chemical_species: list
    physical_metrics: list
    biological_observations: list
    methodology: str
    disciplinary_frame: str
    elapsed_s: float
    parse_error: str = None
    error: str = None

    @property
    def all_metrics(self) -> list:
        return self.chemical_species + self.physical_metrics + self.biological_observations

    @property
    def n_metrics(self) -> int:
        return len(self.all_metrics)

    @property
    def looks_non_biophysical(self) -> bool:
        return self.n_metrics == 0 and self.disciplinary_frame in NON_BIO_FRAMES

def _safe_keywords(row):
    kw = row.get('keywords')
    if pd.notna(kw) and str(kw).strip():
        return str(kw)
    fb = row.get('top_terms_no_stopwords')
    return str(fb) if pd.notna(fb) else ''

def _coerce_list(v):
    if v is None:
        return []
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    return [str(v).strip()] if str(v).strip() else []

def _parse_agent1_json(raw):
    s, e = raw.find('{'), raw.rfind('}')
    if s == -1 or e == -1:
        raise ValueError('no JSON braces in response')
    data = json.loads(raw[s:e+1])
    chem = _coerce_list(data.get('chemical_species'))
    phys = _coerce_list(data.get('physical_metrics'))
    bio = _coerce_list(data.get('biological_observations'))
    meth = str(data.get('methodology', 'none')).strip().lower()
    if meth not in VALID_METHODOLOGY:
        meth = 'none'
    frame = str(data.get('disciplinary_frame', 'other')).strip().lower()
    if frame not in VALID_FRAMES:
        frame = 'other'
    return chem, phys, bio, meth, frame

def _grounding_filter(items, grounding_text_lower):
    """Anti-alucinacion: descarta items que no esten en el abstract."""
    out = []
    for it in items:
        it_l = it.lower().strip()
        if not it_l:
            continue
        if it_l in grounding_text_lower:
            out.append(it); continue
        words = [w for w in re.findall(r'\w+', it_l) if len(w) >= 4]
        if words and any(w in grounding_text_lower for w in words):
            out.append(it)
    return out

def run_agent1(client, model, row):
    abstract_text = str(row.get('clean_abstract') or '')[:MAX_ABSTRACT_CHARS]
    title_text = str(row.get('title') or '')[:500]
    grounding = (title_text + ' ' + abstract_text).lower()

    prompt = AGENT1_PROMPT.format(
        title=title_text,
        journal=str(row.get('journal') or '(unknown)')[:200],
        keywords=_safe_keywords(row)[:500],
        abstract=abstract_text,
    )
    try:
        raw, elapsed = client.generate(model=model, prompt=prompt, json_mode=True, temperature=0.0)
    except Exception as e:
        return Agent1Result('', [], [], [], 'none', 'other', 0.0, error=str(e))
    try:
        chem, phys, bio, meth, frame = _parse_agent1_json(raw)
    except Exception as e:
        return Agent1Result(raw, [], [], [], 'none', 'other', elapsed, parse_error=str(e))

    # Anti-alucinacion: filtra items que no esten en el abstract.
    chem = _grounding_filter(chem, grounding)
    phys = _grounding_filter(phys, grounding)
    bio  = _grounding_filter(bio,  grounding)
    return Agent1Result(raw, chem, phys, bio, meth, frame, elapsed)


### 8.1 Probar Agente 1 sobre 1 paper

Vemos los 5 campos sobre el primer paper validado.

In [ ]:
demo_row = df_sample.iloc[0]
print(f'doc_id   : {demo_row["doc_id"]}')
print(f'title    : {str(demo_row["title"])[:120]}...')
print(f'journal  : {demo_row["journal"]}')
print(f'keywords : {demo_row["keywords"]}')
print(f'abstract : {len(str(demo_row["clean_abstract"]))} chars')

a1 = run_agent1(client, AGENT1_MODEL, demo_row)

print(f'\n--- AGENTE 1 (estructurado) | {a1.elapsed_s:.2f}s ---')
print(f'  chemical_species         ({len(a1.chemical_species)}): {a1.chemical_species}')
print(f'  physical_metrics         ({len(a1.physical_metrics)}): {a1.physical_metrics}')
print(f'  biological_observations  ({len(a1.biological_observations)}): {a1.biological_observations}')
print(f'  methodology              : {a1.methodology}')
print(f'  disciplinary_frame       : {a1.disciplinary_frame}')
print(f'  TOTAL items              : {a1.n_metrics}')
print(f'  vota fast_skip?          : {a1.looks_non_biophysical}  (= 0 items AND frame en {NON_BIO_FRAMES})')

if a1.error:
    print(f'\n[ERROR HTTP] {a1.error}')
elif a1.parse_error:
    print(f'\n[WARN parse] {a1.parse_error}\nRaw: {a1.raw_output[:400]!r}')
else:
    print(f'\n[OK] Extraccion completada.')


## 9. Scoring determinista por *overlap* con `pb_reference.csv`

**Esta es la pieza que el v4 baseline no tiene** — y la que más debería empujar el rendimiento hacia arriba.

### Idea

`pb_reference.csv` define, para cada PB, **6 columnas de keywords** distintas (hasta ahora sólo se usaba `applied_keywords_upv`). Cruzando el texto del paper (title + abstract + top_terms) con cada vocabulario podemos **rankear los PBs candidatos** sin coste de LLM:

| Columna del CSV | Peso | Por qué |
|---|---|---|
| `core_keywords` | **3** | Términos canónicos del PB (ej: PB1 → "climate change", "greenhouse gases", "radiative forcing"). Match aquí es señal fuerte. |
| `extended_keywords` | **2** | Sinónimos y términos secundarios. |
| `applied_keywords_upv` | **1** | Términos del contexto aplicado UPV (ingeniería, sistemas). Más ruidosos, ponderan menos. |

Match con *word boundary* (`\b…\b`) para evitar falsos positivos del tipo "carbón" → "carbon".

### Por qué importa

1. **Es independiente del LLM** → 100 % reproducible, instantáneo, no consume Ollama.
2. **No depende de `keywords` ni `journal`** del corpus. Trabaja sobre el **texto crudo**, que siempre está.
3. Pasamos al 14B una pista del estilo *"top candidatos por vocabulario: PB1 (score 12), PB5 (score 7), PB4 (score 3)"* → el 14B ahorra trabajo en el step de "Candidate Mapping" del prompt v4.

In [ ]:
# Scorer determinista por overlap de keywords con pb_reference.csv.
# No usa LLM. Reproducible. Funciona aunque keywords/journal del corpus esten NaN.

KW_COLS_WEIGHTS = [
    ('core_keywords', 3),
    ('extended_keywords', 2),
    ('applied_keywords_upv', 1),
]
MIN_KW_LEN = 4    # ignoramos keywords de <4 chars (CO, pH... falsos positivos)
MIN_PB_SCORE = 2  # umbral por debajo del cual no consideramos al PB candidato

def _build_pb_keyword_index(df_pbs):
    """Pre-compila las regex por PB (una sola vez). Devuelve dict {pb_code: [(regex, peso, kw)]}."""
    index = {}
    for _, row in df_pbs.iterrows():
        entries = []
        for col, weight in KW_COLS_WEIGHTS:
            val = row.get(col)
            if pd.isna(val):
                continue
            for kw in str(val).split(';'):
                kw = kw.strip().lower()
                if len(kw) < MIN_KW_LEN:
                    continue
                rx = re.compile(r'\b' + re.escape(kw) + r'\b', flags=re.IGNORECASE)
                entries.append((rx, weight, kw))
        index[row['pb_code']] = entries
    return index

PB_KW_INDEX = _build_pb_keyword_index(df_pbs)
print(f'Indice de keywords compilado para {len(PB_KW_INDEX)} PBs.')
print(f'Total entradas (kw, peso) por PB:')
for pb, entries in PB_KW_INDEX.items():
    by_w = {w: sum(1 for _, ww, _ in entries if ww == w) for w in (3, 2, 1)}
    print(f'  {pb}: total={len(entries)}  core(w3)={by_w[3]}  ext(w2)={by_w[2]}  appl(w1)={by_w[1]}')

def score_pb_overlap(text, index=PB_KW_INDEX):
    """
    Devuelve dict {pb_code: {'score': int, 'matches': [(kw, peso), ...]}}
    Todas las matches se cuentan UNA sola vez por keyword (set).
    """
    text_l = (text or '').lower()
    out = {}
    for pb, entries in index.items():
        score = 0
        matches = []
        seen = set()
        for rx, w, kw in entries:
            if kw in seen:
                continue
            if rx.search(text_l):
                score += w
                matches.append((kw, w))
                seen.add(kw)
        out[pb] = {'score': score, 'matches': matches}
    return out

def top_pb_candidates(scores, n=4, min_score=MIN_PB_SCORE):
    """Ordena PBs por score desc y devuelve los top-n con score >= min_score."""
    ranked = sorted(scores.items(), key=lambda x: -x[1]['score'])
    return [(pb, info) for pb, info in ranked[:n] if info['score'] >= min_score]


### 9.1 Scoring sobre el demo paper

Aplicamos el scorer al **mismo paper** del Agente 1. Mostramos también las palabras concretas que dispararon cada match.

In [ ]:
# Construimos el texto a escanear: title + abstract + top_terms_no_stopwords (siempre presentes).
def build_scoring_text(row):
    parts = [str(row.get('title') or ''),
             str(row.get('clean_abstract') or ''),
             str(row.get('top_terms_no_stopwords') or '')]
    return ' '.join(p for p in parts if p and p != 'nan')

scoring_text = build_scoring_text(demo_row)
print(f'Texto escaneado: {len(scoring_text)} chars')

scores = score_pb_overlap(scoring_text)
print('\nScores por PB (todos los 9, ordenados):')
for pb, info in sorted(scores.items(), key=lambda x: -x[1]['score']):
    print(f'  {pb}: score={info["score"]:>3d}  matches={[kw for kw, _ in info["matches"][:6]]}{"..." if len(info["matches"])>6 else ""}')

print('\nTop candidatos (score >= MIN_PB_SCORE):')
top = top_pb_candidates(scores, n=4)
for pb, info in top:
    pretty = ', '.join(f'{kw}(w{w})' for kw, w in info['matches'][:5])
    print(f'  {pb}: {info["score"]} pts  | {pretty}')

if not top:
    print('  (ninguno) → señal "deterministic vota no-biofisico"')


## 10. Agente 3 — Juez experto (qwen2.5:14b) con **3 señales**

El prompt v4 (principle-driven + calibration cases) sigue siendo la base. Le añadimos **dos bloques de pista**:

1. `<agent1_extraction>` — los 5 campos del Agente 1 (químicos / físicos / biológicos / metodología / disciplina).
2. `<deterministic_keyword_scoring>` — el ranking de PBs candidatos del scorer determinista, con las palabras concretas que dispararon cada match.

**Cómo deben usarse las pistas (lo dice el prompt explícitamente):**
- Son **evidencia débil**: el abstract es la única verdad.
- Si las dos pistas coinciden en un PB y el abstract lo confirma → confianza alta.
- Si las pistas se contradicen → ignorarlas y razonar desde el texto.
- Si las pistas no aparecen (caso vacío) → razonar como en el v4 puro.

In [ ]:
# Agente 3: prompt v4 (principle-driven) + dos bloques de pista (extraccion + scoring).

AGENT3_PROMPT_TMPL = """<system_role>
You are an expert scientific evaluator analyzing research abstracts from the Universitat Politecnica de Valencia (UPV) against the Planetary Boundaries (PBs) framework. Your judgment matters more than rigid rule-following. Use the provided framework as a reasoning aid, not a script.
</system_role>

<task>
Identify the Planetary Boundaries that the research actually MEASURES or MODELS, strictly separating real biophysical analysis from mere background or motivational context.
</task>

<instructions>
Follow these analytical steps:
1. Identify the *operational object*: the exact variable being measured, modelled or experimentally manipulated. Quantification, methodology, comparisons or model outputs around a variable indicate it is the focus. Mere mention without numbers or methods is just background.
2. Match with PB: pick the PB whose Core Definition / Activation Logic matches the operational object. The *primary* PB is the one being analytically resolved, not the one mentioned in the introduction.
3. Apply EXCLUSION RULES: if the abstract is purely social, legal, governance, education, philosophy or pure-software theory and biophysical terms are only motivational, the verdict is "None".
4. Common confusions:
   - Aerosols / PM / AOD belong to PB9, not PB1.
   - "Climate" or "warming" alone is NOT enough for PB1. There must be an explicit climate driver, scenario or ecological climate impact being studied.
   - Biodiversity / ecosystem responses are a legitimate primary candidate for PB7.
   - Nutrients (N, P) point to PB4.
   - Water quantity/quality point to PB5. Pick whichever variable the paper actually resolves.
5. Think step-by-step. Output only the high-level summary.
</instructions>

<reference_framework>
{pb_rules}
</reference_framework>

<weak_signals>
The following two blocks come from automatic pre-processing. They are HINTS, not facts. The abstract is the ONLY ground truth. Use the hints to direct your attention; ignore them when they contradict what you read.

  <agent1_extraction>
    A small LLM extracted the following structured fields from the abstract:
    - chemical_species:        {a1_chem}
    - physical_metrics:        {a1_phys}
    - biological_observations: {a1_bio}
    - methodology:             {a1_meth}    (measured > modelled > mentioned > none)
    - disciplinary_frame:      {a1_frame}
  </agent1_extraction>

  <deterministic_keyword_scoring>
    A keyword scorer cross-referenced the title + abstract + top-terms against the official vocabulary in pb_reference.csv. Top PB candidates by overlap score:
    {kw_ranking}
    (Empty list means no PB vocabulary matched the text.)
  </deterministic_keyword_scoring>
</weak_signals>

<calibration_cases>
These cases share one principle: *the primary PB is whatever variable the paper quantifies or models*, regardless of the introduction's rhetorical framing. Use them ONLY to calibrate your judgment, NOT to mimic their wording.

- Case A: A paper studies the central carbon metabolism of anammox bacteria using 13C isotope tracing. The abstract reports specific metabolic pathways and quantified findings. Focus: Nitrogen biogeochemistry (measured, not just mentioned). Primary PB: PB4. Unambiguous.
- Case B: A paper verifies the WRF forecast model comparing initial conditions against radar data. It mentions "changing climate" in the opening. Focus: Forecast-model accuracy. Climate is motivational backdrop, not a measured variable. Primary PB: None.
- Case C: A paper quantifies detection data for 24 mammal species in response to human footprint. Focus: Ecological response of biodiversity to human pressure (measured with statistics). Primary PB: PB7. PB6 (land-system change) is just a contextual driver.
</calibration_cases>

<input_data>
<abstract>
{abstract}
</abstract>
</input_data>

<constraints>
- DO NOT reuse phrases from the calibration cases. Your abstract deserves a fresh reading.
- Write your own reasoning, citing the abstract's own terms.
- DO NOT output any markdown formatting (like ```json).
- DO NOT include any conversational text, greetings or explanations outside the JSON.
</constraints>

<output_format>
Return ONLY valid JSON in the exact structure below:
{{
    "reasoning_process": "A very high-level summary (2-3 sentences). State exactly what is measured/modelled, why you chose this PB, and why it is not background context. Mention briefly whether the weak signals agreed or disagreed with you.",
    "primary_pb": {{"pb_code": "PBX", "confidence": "High/Medium/Low"}} or null,
    "secondary_pbs": [{{"pb_code": "PBY", "confidence": "High/Medium/Low"}}],
    "rejected_pbs": ["PBZ", "PBW"]
}}
</output_format>"""

@dataclass
class Agent3Result:
    raw_output: str
    reasoning: str
    primary_pb: str
    primary_conf: str
    secondary_pbs: str
    rejected_pbs: str
    elapsed_s: float
    error: str = None

def parse_agent3_json(raw_text):
    s, e = raw_text.find('{'), raw_text.rfind('}')
    if s == -1 or e == -1:
        raise ValueError('No JSON object in response')
    data = json.loads(raw_text[s:e+1])
    reasoning = data.get('reasoning_process', '')
    pp = data.get('primary_pb')
    if isinstance(pp, dict):
        prim_code, prim_conf = pp.get('pb_code') or 'None', pp.get('confidence') or 'Unknown'
    else:
        prim_code, prim_conf = 'None', 'N/A'
    sec = data.get('secondary_pbs') or []
    sec_codes = ', '.join(it.get('pb_code', '') for it in sec if isinstance(it, dict)) or 'None'
    rej = data.get('rejected_pbs') or []
    rej_codes = ', '.join(str(x) for x in rej) if rej else 'None'
    return reasoning, prim_code, prim_conf, sec_codes, rej_codes

def format_kw_ranking(top_candidates):
    """Convierte el output de top_pb_candidates en texto para el prompt."""
    if not top_candidates:
        return '    (no PB vocabulary matched)'
    lines = []
    for pb, info in top_candidates:
        kws = ', '.join(f'{kw}(w{w})' for kw, w in info['matches'][:6])
        lines.append(f'    - {pb} (score={info["score"]}): {kws}')
    return '\n'.join(lines)

def run_agent3(client, model, abstract, pb_rules, a1, kw_top):
    """Recibe el Agent1Result completo + top candidatos del scorer."""
    prompt = AGENT3_PROMPT_TMPL.format(
        pb_rules=pb_rules,
        a1_chem=a1.chemical_species or '[]',
        a1_phys=a1.physical_metrics or '[]',
        a1_bio=a1.biological_observations or '[]',
        a1_meth=a1.methodology,
        a1_frame=a1.disciplinary_frame,
        kw_ranking=format_kw_ranking(kw_top),
        abstract=abstract[:MAX_ABSTRACT_CHARS],
    )
    try:
        raw, elapsed = client.generate(model=model, prompt=prompt, json_mode=True, temperature=0.0)
    except Exception as e:
        return Agent3Result('', '', 'Error', 'Error', 'Error', 'Error', 0.0, str(e))
    try:
        reasoning, p, c, s, r = parse_agent3_json(raw)
    except Exception as e:
        return Agent3Result(raw, '', 'ParseError', 'ParseError', 'ParseError', 'ParseError', elapsed, str(e))
    return Agent3Result(raw, reasoning, p, c, s, r, elapsed)


### 10.1 Probar Agente 3 con las 2 señales sobre el demo paper

Reutilizamos `a1` (cell 8.1) y `top` (cell 9.1) y se los pasamos al juez 14B. Comparamos con el ground truth.

In [ ]:
a3 = run_agent3(client, AGENT3_MODEL, str(demo_row['clean_abstract']), PB_RULES, a1, top)

print(f'--- AGENTE 3 (con 3 senales) | {a3.elapsed_s:.2f}s ---')
print(f'  primary_pb     : {a3.primary_pb}  (confidence={a3.primary_conf})')
print(f'  secondary_pbs  : {a3.secondary_pbs}')
print(f'  rejected_pbs   : {a3.rejected_pbs}')
print(f'\nrazonamiento:\n  {a3.reasoning}')

# Comparar con ground truth
gt_row = df_gt[df_gt['doc_id'] == demo_row['doc_id']].iloc[0]
print(f'\n[GT] 1stpb={gt_row["1stpb_label"]}  2ndpb={gt_row["2ndpb"]}  3rdpb={gt_row["3rdpb"]}')

# Resumen de senales para auditar acuerdo/desacuerdo
print('\n=== Resumen de las 3 senales para este paper ===')
print(f'  Agent 1 dice            : {a1.n_metrics} items, frame={a1.disciplinary_frame}, methodology={a1.methodology}')
print(f'  Scorer determinista     : top = {[pb for pb, _ in top]}')
print(f'  Juez 14B (con pistas)   : primary = {a3.primary_pb}')
print(f'  Ground truth humano     : {gt_row["1stpb_label"]}')


## 11. Pipeline con **router por consenso 3-de-3**

Regla de skip (mucho más conservadora que la versión anterior):

> Sólo se hace `fast_skip → None` si **los TRES indicadores** votan "no biofísico":
> 1. Agente 1 extrajo **0 items** Y `disciplinary_frame ∈ {social_sciences, economics, law_policy, education}`.
> 2. Scorer determinista: **score top == 0** (ningún PB tiene match en su vocabulario).
> 3. (Implícito) — el 14B ni siquiera se llama porque ya hay consenso.

En cualquier otro caso, el 14B se ejecuta con las 2 señales como pista.

**Por qué este diseño no puede empeorar el baseline:**
- El consenso 3-de-3 es muy restrictivo → casi todos los papers van al 14B.
- El 14B siempre recibe abstract crudo + las 2 pistas → tiene **estrictamente más información** que el v4 puro.
- Los falsos negativos del filtro requieren que A1 + scorer fallen *ambos* a la vez en un paper biofísico → escenario raro.

In [ ]:
def consensus_vote_skip(a1, kw_top):
    """True solo si Agente 1 Y scorer determinista coinciden en 'no biofisico'."""
    a1_says_no = a1.looks_non_biophysical
    kw_says_no = (len(kw_top) == 0)
    return a1_says_no and kw_says_no

def run_pipeline(client, df, pb_rules, agent1_model, agent3_model, verbose=True):
    rows = []
    for i, (_, r) in enumerate(df.iterrows(), 1):
        t0 = time.perf_counter()

        # Senal 1: Agente 1
        a1 = run_agent1(client, agent1_model, r)

        # Senal 2: scorer determinista (gratis, no usa LLM)
        scoring_text_i = build_scoring_text(r)
        kw_scores = score_pb_overlap(scoring_text_i)
        kw_top_i = top_pb_candidates(kw_scores, n=4)

        rec = {
            'doc_id': r['doc_id'],
            'agent1_chemical': '; '.join(a1.chemical_species),
            'agent1_physical': '; '.join(a1.physical_metrics),
            'agent1_biological': '; '.join(a1.biological_observations),
            'agent1_methodology': a1.methodology,
            'agent1_frame': a1.disciplinary_frame,
            'agent1_n_items': a1.n_metrics,
            'agent1_time_s': round(a1.elapsed_s, 2),
            'agent1_error': a1.error or '',
            'agent1_parse_error': a1.parse_error or '',
            'kw_top_pbs': '; '.join(f'{pb}({info["score"]})' for pb, info in kw_top_i),
            'kw_top_score': kw_top_i[0][1]['score'] if kw_top_i else 0,
        }

        # Router por consenso 3-de-3
        if consensus_vote_skip(a1, kw_top_i):
            rec.update(route='fast_skip_consensus',
                       llm_primary_pb='None', llm_primary_conf='ConsensusFilter',
                       llm_secondary_pbs='None', llm_rejected_pbs='None',
                       llm_reasoning='Consensus: Agent1 (0 items, non-bio frame) + scorer (no PB matched).',
                       agent3_time_s=0.0, agent3_error='')
        else:
            a3 = run_agent3(client, agent3_model, str(r['clean_abstract']), pb_rules, a1, kw_top_i)
            rec.update(route='llm_judged',
                       llm_primary_pb=a3.primary_pb, llm_primary_conf=a3.primary_conf,
                       llm_secondary_pbs=a3.secondary_pbs, llm_rejected_pbs=a3.rejected_pbs,
                       llm_reasoning=a3.reasoning,
                       agent3_time_s=round(a3.elapsed_s, 2), agent3_error=a3.error or '')

        rec['total_time_s'] = round(time.perf_counter() - t0, 2)
        rows.append(rec)
        if verbose:
            print(f'[{i}/{len(df)}] {rec["doc_id"]} route={rec["route"]:22s} '
                  f'a1_items={rec["agent1_n_items"]} kw_top={rec["kw_top_pbs"][:30] or "(none)":30s} '
                  f'a3={rec["agent3_time_s"]}s pb={rec["llm_primary_pb"]}')
    return pd.DataFrame(rows)

N_DEMO = 10
df_demo = df_sample.head(N_DEMO).copy()
print(f'Ejecutando cascada de enriquecimiento sobre {len(df_demo)} papers...\n')
df_results = run_pipeline(client, df_demo, PB_RULES, AGENT1_MODEL, AGENT3_MODEL)
df_results


## 12. Análisis del run

### 12.1 Coste y distribución de rutas

In [ ]:
print('Distribucion de rutas:')
print(df_results['route'].value_counts().to_string())
print()

skip_pct = 100 * (df_results['route'] == 'fast_skip_consensus').mean()
avg_a1 = df_results['agent1_time_s'].mean()
judged_mask = df_results['route'] == 'llm_judged'
avg_a3 = df_results.loc[judged_mask, 'agent3_time_s'].mean() if judged_mask.any() else 0
avg_total = df_results['total_time_s'].mean()
baseline_total = avg_a3  # baseline v4 = solo el 14B sobre todos los papers

print(f'fast_skip pct          : {skip_pct:.1f}%')
print(f'tiempo medio Agent 1   : {avg_a1:.2f}s')
print(f'tiempo medio Agent 3   : {avg_a3:.2f}s (sólo en los llm_judged)')
print(f'tiempo medio cascada   : {avg_total:.2f}s/paper')
print(f'tiempo baseline 14B    : ~{baseline_total:.2f}s/paper')
if baseline_total > 0:
    saving = 100 * (1 - avg_total / baseline_total)
    print(f'Ahorro vs baseline     : {saving:+.1f}%  ({"+" if saving>=0 else ""}sentido positivo = mas rapido)')
print('\n(El verdadero valor de este pipeline NO esta en el ahorro sino en la calidad — ver 12.2)')


### 12.2 Calibración contra ground truth

Tres preguntas:

1. **¿El consenso protegió de falsos negativos?** Cualquier paper con GT ≠ None que el cascada marca None es un fallo crítico (recall perdido).
2. **¿La pista del scorer correlaciona con el PB humano?** Si el top-1 del scorer = GT con frecuencia alta, el scorer aporta señal real al 14B.
3. **¿El Agente 1 caracteriza bien la disciplina?** Útil para entender en qué tipo de papers el extractor falla.

In [ ]:
merged = df_results.merge(df_gt[['doc_id', '1stpb_label']], on='doc_id', how='left')
merged['gt_is_none'] = merged['1stpb_label'] == 'None'
merged['llm_is_none'] = merged['llm_primary_pb'] == 'None'

# (1) Matriz de confusion sobre la clase None
print('Matriz de confusion sobre la clase "None":')
ct = pd.crosstab(merged['gt_is_none'], merged['llm_is_none'],
                 rownames=['GT_None'], colnames=['LLM_None'])
print(ct)

tp = ((merged['gt_is_none']) & (merged['llm_is_none'])).sum()
fp = ((~merged['gt_is_none']) & (merged['llm_is_none'])).sum()
fn = ((merged['gt_is_none']) & (~merged['llm_is_none'])).sum()
prec = tp / (tp + fp) if (tp + fp) else float('nan')
rec  = tp / (tp + fn) if (tp + fn) else float('nan')
print(f'\nPrecision "None"  : {prec:.2%}')
print(f'Recall    "None"  : {rec:.2%}')
print(f'Falsos rechazos (caros): {fp}')

# (2) Calidad de la senal del scorer determinista
print('\n--- Senal scorer determinista vs GT ---')
def kw_top1(s):
    if not s or pd.isna(s): return None
    first = s.split(';')[0].strip()
    return first.split('(')[0] if '(' in first else first
merged['kw_top1'] = merged['kw_top_pbs'].apply(kw_top1)
merged_with_pb = merged[~merged['gt_is_none']].copy()
if len(merged_with_pb):
    hit = (merged_with_pb['kw_top1'] == merged_with_pb['1stpb_label']).mean()
    print(f'Hit@1 del scorer (sobre los {len(merged_with_pb)} papers con GT real): {hit:.2%}')
    print('  (un 0% no significa que el scorer sea inutil — es solo una pista para el 14B)')

# (3) Caracterizacion disciplinar
print('\n--- Disciplinary frame del Agente 1 ---')
print(merged.groupby(['agent1_frame', 'gt_is_none']).size().unstack(fill_value=0))


In [ ]:
# Hit@1 del juez 14B (donde el LLM dio asignacion).
evaluable = merged[merged['llm_primary_pb'].str.startswith('PB')].copy()
if len(evaluable):
    evaluable['hit'] = evaluable['llm_primary_pb'] == evaluable['1stpb_label']
    print(f'Hit@1 del juez 14B (sobre {len(evaluable)} papers con asignacion): {evaluable["hit"].mean():.2%}')
    print('\nDesglose por paper:')
    cols = ['doc_id', 'agent1_frame', 'agent1_n_items', 'kw_top_pbs', 'llm_primary_pb', '1stpb_label', 'hit']
    print(evaluable[cols].to_string(index=False))
else:
    print('Ningun paper de la muestra recibio asignacion de PB (todos cayeron en fast_skip o error).')


## 13. Próximos pasos

1. **Run completo sobre los 208 validados** (calibración seria contra ground truth):
   ```bash
   python nlp/llm/runners/pipeline_agentes.py --limit 208 \
       --output nlp/llm/outputs/pipeline_cascada/cal_208_v2.csv
   ```

2. **Compara contra el baseline v4 puro** (mismo subset de 208 papers):
   - Hit@1 cascada vs Hit@1 v4 → ¿el enriquecimiento ayuda al juez?
   - Recall sobre `None` → ¿el consenso evita los falsos rechazos?

3. **Si el cascada NO mejora al baseline en Hit@1**: el problema es que las pistas no aportan información nueva. Diagnóstico:
   - Mira los casos donde `kw_top1 != llm_primary_pb`: ¿el 14B ignoró el scorer? ¿O el scorer estaba mal?
   - Si el 14B ignora siempre las pistas, **endurecer el prompt** para que las cite explícitamente en el `reasoning_process`.

4. **Si el cascada mejora al baseline pero la mejora es menor del 2 %**: probablemente el 14B v4 ya estaba en su techo. Considera escalar el corpus en vez del modelo.

5. **Auditoría de los 5 % peores casos**: papers donde Hit@1 falla a pesar del consenso. Suelen ser papers con métricas ambiguas (ej. CO2 en contexto económico → ¿PB1 o None?).